# 04 — Domain Adaptation (Contribution 3)
**DeepLense GSoC 2026 — Pallab Mondal**

This notebook demonstrates the ADDA (Adversarial Discriminative Domain Adaptation) pipeline:

1. Load a pre-trained LensPINN (source model, trained on Model_I simulated data)
2. Set up the target encoder as a copy of the source encoder
3. Run the adversarial adaptation loop to align feature spaces
4. Evaluate on target domain (Model_IV or HSC real data)

**Why physics-informed features help:**  
The physics encoder produces features grounded in the lensing equation (Einstein radius, deflection angle).  
These are *domain-invariant by construction* — the physics is the same in simulations and real HSC images.

In [ ]:
import sys, os
from pathlib import Path

MY_WORK = Path(os.getcwd()).parent
sys.path.insert(0, str(MY_WORK))

import torch
import torch.nn as nn
import copy
import numpy as np
import matplotlib.pyplot as plt

from config import (
    MODEL_I_TRAIN, MODEL_I_TEST,
    CLASS_NAMES, IMAGE_SIZE, BATCH_SIZE, CHECKPOINTS_DIR,
)
from utils.data_loader import build_dataloaders, DeepLenseDataset, get_val_transform
from models.lens_pinn  import LensPINN
from models.adda       import DomainDiscriminator, ADDATrainer
from utils.metrics     import evaluate_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ── 1.  Load source model ─────────────────────────────────────────────────────
# Either load a trained checkpoint or create one for demo:

SOURCE_CKPT = CHECKPOINTS_DIR / 'lens_pinn_model_i_best.pt'

source_model = LensPINN(num_classes=3).to(device)

if SOURCE_CKPT.exists():
    ckpt = torch.load(SOURCE_CKPT, map_location=device, weights_only=False)
    source_model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded checkpoint: {SOURCE_CKPT}')
else:
    print('No checkpoint found — using randomly initialised weights for demo.')
    print('Train with: python train.py --model lens_pinn --dataset model_i')

In [ ]:
# ── 2.  Prepare source & target dataloaders ───────────────────────────────────
# Source: Model I simulated data (labelled)
source_train, _, source_test = build_dataloaders(
    train_root=MODEL_I_TRAIN,
    test_root=MODEL_I_TEST,
    class_names=CLASS_NAMES[:3],
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

# Target: Model IV (real HSC hybrid) — update this path when downloaded
# MODEL_IV_DIR = Path('../Data/Model_IV')
# For now we use Model III as a proxy target domain
from config import MODEL_III_TRAIN, MODEL_III_TEST

target_train, target_val, target_test = build_dataloaders(
    train_root=MODEL_III_TRAIN,
    test_root=MODEL_III_TEST,
    class_names=CLASS_NAMES[:3],
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

print(f'Source train batches : {len(source_train)}')
print(f'Target train batches : {len(target_train)}')
print(f'Target val   batches : {len(target_val)}')

In [ ]:
# ── 3.  Set up ADDA ───────────────────────────────────────────────────────────
# Clone source encoder as target encoder
target_encoder = copy.deepcopy(source_model)

# We adapt the FULL model (encoder + physics layers + CNN) as a single unit.
# The discriminator receives the logit vector (dim = num_classes = 3 here).
# For a cleaner separation, you'd expose the CNN feature extractor;
# for simplicity this demo uses logits as the domain-alignment signal.

FEATURE_DIM = 3  # logits dimension for 3-class (change to match encoder hidden dim for cleaner ADDA)

discriminator = DomainDiscriminator(feature_dim=FEATURE_DIM, hidden_dim=64)

# Use a shallow adapter to extract a single feature vector from the model
# (here we reuse the full model and attach via _encode_source override)

trainer = ADDATrainer(
    source_encoder=source_model,
    target_encoder=target_encoder,
    classifier=nn.Identity(),  # logits ARE the feature here
    discriminator=discriminator,
    device=device,
)

print('ADDA trainer ready.')

In [ ]:
# ── 4.  Baseline evaluation (before adaptation) ───────────────────────────────
print('=== Source model on source test set ===')
evaluate_model(source_model, source_test, device, class_names=CLASS_NAMES[:3])

print('\n=== Source model on target test set (no adaptation) ===')
evaluate_model(source_model, target_test, device, class_names=CLASS_NAMES[:3])

In [ ]:
# ── 5.  ADDA adaptation ───────────────────────────────────────────────────────
# (demo: 3 epochs; use 50+ for real runs)
# trainer.adapt(
#     source_loader=source_train,
#     target_loader=target_train,
#     target_loader_val=target_val,
#     epochs=3,
#     lr_target=1e-5,
#     lr_disc=1e-4,
#     save_path=str(CHECKPOINTS_DIR / 'adda_best.pt'),
# )

# Uncomment the above when running a real experiment.
# The adapt() call is commented out here to ensure the notebook doesn't block.

print('ADDA loop setup demonstrated. Uncomment `trainer.adapt(...)` to run.')

In [ ]:
# ── 6.  t-SNE feature visualisation: source vs target before/after ────────────
from sklearn.manifold import TSNE

@torch.no_grad()
def extract_features(model, loader, max_batches=20):
    feats, labels = [], []
    for i, (imgs, lbs) in enumerate(loader):
        if i >= max_batches:
            break
        imgs = imgs.to(device)
        out = model(imgs)
        if isinstance(out, (tuple, list)):
            logits = out[-1]
        else:
            logits = out
        feats.append(logits.cpu().numpy())
        labels.append(lbs.numpy())
    return np.concatenate(feats), np.concatenate(labels)

src_feats,  src_labels  = extract_features(source_model, source_test)
tgt_feats,  tgt_labels  = extract_features(source_model, target_test)

# Combine and colour by DOMAIN (not class)
all_feats  = np.concatenate([src_feats, tgt_feats])
domain_lbl = np.array([0]*len(src_feats) + [1]*len(tgt_feats))

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embedded = tsne.fit_transform(all_feats)

fig, ax = plt.subplots(figsize=(7, 6))
colors = ['#4C9BE8', '#E86C4C']
for d, label, c in zip([0, 1], ['Source (sim)', 'Target (real)'], colors):
    mask = domain_lbl == d
    ax.scatter(embedded[mask, 0], embedded[mask, 1],
               c=c, label=label, alpha=0.5, s=8)
ax.set(title='t-SNE: Source vs Target features (before ADDA)', xlabel='t-SNE 1', ylabel='t-SNE 2')
ax.legend()
plt.tight_layout()
plt.show()

print('\nAfter ADDA, source and target clusters should overlap.')